In [139]:
import numpy as np
import polars as pl
import seaborn as sns

In [140]:
infpath = '/mnt/data/Max/repos/reverlor/param_selection/test_repeat_detection/data/find_repeats_results/merged.tsv'

In [141]:
df = pl.read_csv(infpath, separator='\t')

FileNotFoundError: No such file or directory (os error 2): ...epos/reverlor/param_selection/test_repeat_detection/data/find_repeats_results/merged.tsv (set POLARS_VERBOSE=1 to see full path)

In [ ]:
df = df.with_columns(
    (
        pl.col('repeat_len').cast(pl.String) \
        + '_' \
        + pl.col('repeat_idx').cast(pl.String) \
        + '_' \
        + pl.col('rate').cast(pl.String) \
    ).alias('repeat_id')
)

In [ ]:
df.head()

In [ ]:
true_df = df.filter(
    pl.col('value_type') == 'true'
)
pred_df = df.filter(
    pl.col('value_type') == 'pred'
)

In [ ]:
true_df.shape

In [ ]:
pred_df.shape

In [ ]:
true_df.filter(pl.col('rate') == 0.0).head()

In [ ]:
tyk_repeat_id = '10000_0_0.0'

In [ ]:
true_df.filter(
    (pl.col('repeat_id') == tyk_repeat_id)
)

In [ ]:
pred_df.filter(
    (pl.col('repeat_id') == tyk_repeat_id)
)

In [ ]:
pred_df.filter(
    (pl.col('repeat_id') == tyk_repeat_id) \
  & ( (pl.col('start_coord') - 510209).abs() < 10 ) \
  & ( (pl.col('end_coord')   - 520209).abs() < 10 )
)

In [ ]:
pred_df.filter(
    (pl.col('repeat_id') == tyk_repeat_id) \
  & ( (pl.col('start_coord') - 483432).abs() < 10 ) \
  & ( (pl.col('end_coord')   - 493432).abs() < 10 )
)

In [ ]:
all_repeat_ids = frozenset(
    true_df['repeat_id']
)

In [ ]:
MAX_ERROR_BP = 10

In [ ]:
repeat_id_list = list()
repeat_start_coord_list = list()
repeat_found_list = list()

for repeat_id in all_repeat_ids:
    curr_true_df = true_df.filter(pl.col('repeat_id') == repeat_id)
    curr_pred_df = pred_df.filter(pl.col('repeat_id') == repeat_id)

    for true_row in curr_true_df.to_dicts():
        curr_pred_detect_df = curr_pred_df.filter(
            (pl.col('repeat_id') == repeat_id) \
          & ( (pl.col('start_coord') - true_row['start_coord']).abs() <= MAX_ERROR_BP ) \
          & ( (pl.col('end_coord')   - true_row['end_coord']).abs() <= MAX_ERROR_BP )
        )

        found_value = 0
        if curr_pred_detect_df.shape[0] > 0:
            found_value = 1
        # end if

        repeat_id_list.append(repeat_id)
        repeat_start_coord_list.append(true_row['start_coord'])
        repeat_found_list.append(found_value)
    # end for
# end for

In [ ]:
print(len(repeat_id_list))
print(len(repeat_start_coord_list))
print(len(repeat_found_list))

repeat_detect_df = pl.DataFrame({
    'repeat_id': repeat_id_list,
    'repeat_start_coord': repeat_start_coord_list,
    'repeat_detected': repeat_found_list,
})

In [ ]:
repeat_detect_df.shape

In [ ]:
assert repeat_detect_df.shape[0] == true_df.shape[0]

In [ ]:
repeat_detect_df = repeat_detect_df.join(
    true_df,
    on='repeat_id',
    how='inner'
).select(
    pl.col('repeat_id', 'repeat_start_coord', 'repeat_detected', 'repeat_len', 'repeat_idx', 'rate')
).unique()

In [ ]:
repeat_detect_df.shape

In [ ]:
repeat_detect_df.head()

In [ ]:
N_repeats = true_df.filter(
    (pl.col('repeat_len') == 200) \
    & (pl.col('rate') == 0.0)
).shape[0]
print('N_repeats = {}'.format(N_repeats))

per_rate_len_summary_df = repeat_detect_df.group_by('rate', 'repeat_len').agg(
    pl.col('repeat_detected').sum().alias('repeat_detected_count')
).sort(by=['rate', 'repeat_len'])

per_rate_len_summary_df = per_rate_len_summary_df.with_columns(
    (pl.col('repeat_detected_count') / N_repeats * 100.0).alias('repeat_detected_percent')
)

per_rate_len_summary_df.head(10)

In [ ]:
df.filter(
    (pl.col('rate') == 0.0) & (pl.col('repeat_len') == 1000) & (pl.col('repeat_idx') == 2)
).sort(by=['repeat_idx', 'value_type', 'start_coord'])

In [ ]:
repeat_detect_df.filter(
    (pl.col('rate') == 0.0) & (pl.col('repeat_len') == 1000) & (pl.col('repeat_detected') != 1)
).sort(by='repeat_idx')

In [ ]:
N_repeats = true_df.filter(
    pl.col('rate') == 0.0
).shape[0]
print('N_repeats = {}'.format(N_repeats))

per_rate_summary_df = repeat_detect_df.group_by('rate').agg(
    pl.col('repeat_detected').sum().alias('repeat_detected_count')
).sort(by=['rate'])

per_rate_summary_df = per_rate_summary_df.with_columns(
    (pl.col('repeat_detected_count') / N_repeats * 100.0).alias('repeat_detected_percent')
)

per_rate_summary_df.head(10)

In [ ]:
N_repeats = true_df.filter(
    (pl.col('repeat_len') == 127) \
    & (pl.col('rate') == 0.0)
).shape[0]
print('N_repeats = {}'.format(N_repeats))

per_rate_len127_summary_df = repeat_detect_df.filter(
    pl.col('repeat_len') == 127
).group_by('rate').agg(
    pl.col('repeat_detected').sum().alias('repeat_detected_count')
).sort(by=['rate'])

per_rate_len127_summary_df = per_rate_len127_summary_df.with_columns(
    (pl.col('repeat_detected_count') / N_repeats * 100.0).alias('repeat_detected_percent')
)

per_rate_len127_summary_df.head(10)

In [ ]:
N_repeats = true_df.filter(
    (pl.col('repeat_len') == 200) \
    & (pl.col('rate') == 0.0)
).shape[0]
print('N_repeats = {}'.format(N_repeats))

per_rate_len200_summary_df = repeat_detect_df.filter(
    pl.col('repeat_len') == 200
).group_by('rate').agg(
    pl.col('repeat_detected').sum().alias('repeat_detected_count')
).sort(by=['rate'])

per_rate_len200_summary_df = per_rate_len200_summary_df.with_columns(
    (pl.col('repeat_detected_count') / N_repeats * 100.0).alias('repeat_detected_percent')
)

per_rate_len200_summary_df.head(10)

In [ ]:
N_repeats = true_df.filter(
    (pl.col('repeat_len') == 10000) \
    & (pl.col('rate') == 0.0)
).shape[0]
print('N_repeats = {}'.format(N_repeats))

per_rate_len10000_summary_df = repeat_detect_df.filter(
    pl.col('repeat_len') == 10000
).group_by('rate').agg(
    pl.col('repeat_detected').sum().alias('repeat_detected_count')
).sort(by=['rate'])

per_rate_len10000_summary_df = per_rate_len10000_summary_df.with_columns(
    (pl.col('repeat_detected_count') / N_repeats * 100.0).alias('repeat_detected_percent')
)

per_rate_len200_summary_df.head(10)

In [ ]:
sns.lineplot(
    per_rate_summary_df,
    x='rate',
    y='repeat_detected_percent'
)

In [ ]:
sns.lineplot(
    per_rate_len127_summary_df,
    x='rate',
    y='repeat_detected_percent'
)

In [ ]:
sns.lineplot(
    per_rate_len200_summary_df,
    x='rate',
    y='repeat_detected_percent'
)

In [ ]:
sns.lineplot(
    per_rate_len10000_summary_df,
    x='rate',
    y='repeat_detected_percent'
)